In [19]:
import re
from datasets import load_dataset
import heapq
import os
import unicodedata
from collections import defaultdict
import regex
from tqdm import tqdm
import pickle
import time
import tracemalloc
import json

### Pretokenization

In [20]:
# Pretokenization Regexes

GPT2_PRE_TOKEN_REGEX = (
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

# Medical regex: keeps IL-6, COVID-19, p53, BRCA1, HbA1c, 25mg, 0.05 intact
MEDICAL_PRE_TOKEN_REGEX = (
    r"""(?:\d+[\-\.]\d+(?:[a-zA-Z]+)?)|"""
    r"""(?:[A-Za-z]+\d+[A-Za-z]*)|"""
    r"""(?:[A-Z]{2,}(?:[-]\d+)+)|"""
    r"""(?:\d+(?:\.\d+)?\s?(?:mg|mL|ug|kg|mmol|umol|nmol|IU|mM|nM))|"""
    r"""'(?:[sdmt]|ll|ve|re)|"""
    r""" ?\p{L}+(?:[-]\p{L}+)*|"""
    r""" ?\p{N}+|"""
    r""" ?[^\s\p{L}\p{N}]+|"""
    r"""\s+(?!\S)|\s+"""
)

MEDICAL_SPECIAL_TOKENS = ["<|endoftext|>", ]

# Text Normalization

_GREEK_MAP = {
    "α": "alpha", "β": "beta",  "γ": "gamma",  "δ": "delta",
    "ε": "epsilon","ζ": "zeta", "η": "eta",    "θ": "theta",
    "κ": "kappa",  "λ": "lambda","μ": "u",     "ν": "nu",
    "ξ": "xi",     "π": "pi",   "ρ": "rho",   "σ": "sigma",
    "τ": "tau",    "φ": "phi",  "χ": "chi",   "ψ": "psi",
    "ω": "omega",  "Α": "Alpha","Β": "Beta",  "Γ": "Gamma",
    "Δ": "Delta",  "Κ": "Kappa","Λ": "Lambda","Π": "Pi",
    "Σ": "Sigma",  "Ω": "Omega",
}

_SYMBOL_MAP = {
    "±": "+/-",  "≥": ">=",    "≤": "<=",   "×": "x",
    "÷": "/",    "·": ".",     "−": "-",    "–": "-",
    "—": "-",    "\u00b0": " degrees ", "\u2019": "'",
    "\u201c": '"', "\u201d": '"',
}

In [21]:
def normalize_medical_text(text: str) -> str:
    """
    Normalize PubMed text before BPE training.
    """
    text = unicodedata.normalize("NFC", text)
    for greek, ascii_eq in _GREEK_MAP.items():
        text = text.replace(greek, ascii_eq)
    for symbol, repl in _SYMBOL_MAP.items():
        text = text.replace(symbol, repl)
    text = regex.sub(r"\\[a-zA-Z]+", "", text)          # LaTeX
    text = (text.replace("&lt;", "<").replace("&gt;", ">")
                .replace("&amp;", "&").replace("&nbsp;", " ")
                .replace("&#160;", " "))                 # HTML entities
    text = regex.sub(r" {2,}", " ", text)
    text = regex.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [22]:
def is_medical_text(text: str, min_length: int = 100) -> bool:
    """
    Filter out DNA sequences, junk data, and non-medical content.
    """
    if len(text) < min_length:
        return False
    
    # Reject DNA sequences (ATCG patterns)
    dna_ratio = len(re.findall(r'[ATCG]', text)) / len(text)
    if dna_ratio > 0.5:  # >50% DNA bases = likely sequence
        return False
    
    # Reject protein sequences (long amino acid strings)
    protein_chars = set('ACDEFGHIKLMNPQRSTVWY')
    if sum(1 for c in text if c in protein_chars) / len(text) > 0.6:
        return False

    
    # Accept if medical keywords
    medical_keywords = {
        'patient', 'treatment', 'disease', 'clinical', 'study',
        'diagnosis', 'therapy', 'medical', 'hospital', 'doctor',
        'symptom', 'drug', 'medicine', 'cancer', 'heart', 'brain',
        'infection', 'surgery', 'outcome', 'method', 'result',
        'abstract', 'conclusion', 'background', 'objective',
    }
    text_lower = text.lower()
    keyword_count = sum(1 for kw in medical_keywords if kw in text_lower)
    
    return keyword_count >= 2  # least 2 keywords   

In [23]:
def create_filtered_medical_corpus(
    dataset,
    output_path: str,
    text_field: str = "article",
    normalize: bool = True,
    boundary_token: str = "<|endoftext|>",
    sample_percent: float = 100,
) -> None:
    
    written = 0
    skipped = 0
    
    with open(output_path, "w", encoding="utf-8") as f:
        for item in tqdm(dataset, desc="Writing corpus"):
            text = item[text_field]
            
            # Filter step
            if not is_medical_text(text):
                skipped += 1
                continue
            
            if normalize:
                text = normalize_medical_text(text)
            
            if text.strip():
                f.write(text)
                f.write(f"\n{boundary_token}\n")
                written += 1

In [24]:
# Heap Helper

class ReversedBytes:
    """
    Reverses byte comparison so Python's min-heap surfaces
    lexicographically LARGEST bytes on frequency ties.
    """
    __slots__ = ("data",)

    def __init__(self, data: bytes):
        self.data = data

    def __lt__(self, other):
        return self.data > other.data

    def __le__(self, other):
        return self.data >= other.data

    def __eq__(self, other):
        return isinstance(other, ReversedBytes) and self.data == other.data

In [25]:
# Pretokenization

def pretokenize(
    chunk: str,
    special_tokens: list[str],
    use_medical_regex: bool = True,
) -> dict[tuple[int, ...], int]:
    """
    Pretokenize a chunk into byte-tuple → frequency mapping.
    Splits on special tokens first so they never enter the regex.
    """
    pattern = MEDICAL_PRE_TOKEN_REGEX if use_medical_regex else GPT2_PRE_TOKEN_REGEX
    special_set = set(special_tokens)

    if special_tokens:
        escaped = [regex.escape(st) for st in special_tokens]
        parts = regex.split(f"({'|'.join(escaped)})", chunk)
        text_parts = [p for p in parts if p not in special_set and p]
    else:
        text_parts = [chunk]

    freqs: dict[tuple[int, ...], int] = {}
    for part in text_parts:
        for m in regex.finditer(pattern, part):
            key = tuple(m.group().encode("utf-8"))
            freqs[key] = freqs.get(key, 0) + 1
    return freqs

### Training BPE

In [26]:
# Main Training Function

def train_medical_bpe_tokenizer(
    input_path: str,
    vocab_size: int = 32_000,
    special_tokens: list[str] | None = None,
    use_medical_regex: bool = True,
    heap_compaction_ratio: float = 3.0,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    """
    Train a BPE tokenizer for medical/PubMed text.
    """
    if special_tokens is None:
        special_tokens = MEDICAL_SPECIAL_TOKENS

    corpus_size = os.path.getsize(input_path)
    print(f"Corpus: {corpus_size / 1024 / 1024:.1f} MB")

    with open(input_path, "r", encoding="utf-8", errors="ignore") as f:
        chunk = f.read()
    word_freq_raw = pretokenize(chunk, special_tokens, use_medical_regex)

    # ── Phase 2: Initialize Vocab & Structures ─────────────────────────
    vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}

    words: dict[int, list[int]] = {}
    word_freqs: dict[int, int] = {}
    for wid, (wt, freq) in enumerate(word_freq_raw.items()):
        words[wid] = list(wt)
        word_freqs[wid] = freq

    for st in special_tokens:
        vocab[len(vocab)] = st.encode("utf-8")

    num_merges = vocab_size - len(vocab)
    if num_merges <= 0:
        raise ValueError(
            f"vocab_size={vocab_size} must exceed initial size={len(vocab)} "
            f"(256 bytes + {len(special_tokens)} special tokens)"
        )
    merges: list[tuple[bytes, bytes]] = []
    print(f"Initial vocab: {len(vocab)} | Merges to do: {num_merges:,}")

    # ── Phase 3: Pair Frequencies + Inverted Index ─────────────────────
    pair_frequencies: dict[tuple[int, int], int] = defaultdict(int)
    pair_to_words: dict[tuple[int, int], set[int]] = defaultdict(set)

    for wid, tokens in words.items():
        freq = word_freqs[wid]
        for i in range(len(tokens) - 1):
            p = (tokens[i], tokens[i + 1])
            pair_frequencies[p] += freq
            pair_to_words[p].add(wid)

    print(f"Unique pairs: {len(pair_frequencies):,}")

    # ── Phase 4: Build Heap ────────────────────────────────────────────
    def make_entry(pair):
        freq = pair_frequencies[pair]
        lex = (ReversedBytes(vocab[pair[0]]), ReversedBytes(vocab[pair[1]]))
        return (-freq, lex, pair)

    heap = [make_entry(p) for p in pair_frequencies]
    heapq.heapify(heap)
    valid_pairs: set[tuple[int, int]] = set(pair_frequencies.keys())

    # ── Phase 5: Merge Loop ────────────────────────────────────────────
    for _ in tqdm(range(num_merges), desc="BPE merges", unit="merge"):

        # Find best valid pair (lazy deletion)
        best_pair = None
        while heap:
            neg_freq, _lex, pair = heapq.heappop(heap)
            if pair not in valid_pairs:
                continue
            cur_freq = pair_frequencies.get(pair, 0)
            if cur_freq != -neg_freq:
                if cur_freq > 0:
                    heapq.heappush(heap, make_entry(pair))
                continue
            best_pair = pair
            break

        if best_pair is None:
            print("No more pairs — stopping early.")
            break

        # Create merged token
        new_id = len(vocab)
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
        merges.append((vocab[best_pair[0]], vocab[best_pair[1]]))

        # Update only affected words via inverted index
        for wid in list(pair_to_words.get(best_pair, set())):
            tokens = words[wid]
            freq = word_freqs[wid]

            # A: Remove old pair counts
            for i in range(len(tokens) - 1):
                p = (tokens[i], tokens[i + 1])
                pair_frequencies[p] -= freq
                if pair_frequencies[p] <= 0:
                    del pair_frequencies[p]
                    valid_pairs.discard(p)
                pair_to_words[p].discard(wid)
                # FIX 1: clean up empty sets → prevents memory leak
                if p in pair_to_words and not pair_to_words[p]:
                    del pair_to_words[p]

            # B: Apply merge
            new_tokens, i = [], 0
            while i < len(tokens):
                if (i < len(tokens) - 1
                        and tokens[i] == best_pair[0]
                        and tokens[i + 1] == best_pair[1]):
                    new_tokens.append(new_id)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            words[wid] = new_tokens

            # C: Add new pair counts
            for i in range(len(new_tokens) - 1):
                p = (new_tokens[i], new_tokens[i + 1])
                pair_frequencies[p] += freq
                valid_pairs.add(p)
                pair_to_words[p].add(wid)
                heapq.heappush(heap, make_entry(p))

        # Clean up merged pair
        valid_pairs.discard(best_pair)
        # FIX 2: explicit pop prevents pair_frequencies/valid_pairs desync
        pair_frequencies.pop(best_pair, None)
        if best_pair in pair_to_words:
            del pair_to_words[best_pair]

        # Heap compaction — keep stale entries under control
        if len(heap) > heap_compaction_ratio * len(valid_pairs):
            heap = [make_entry(p) for p in valid_pairs if p in pair_frequencies]
            heapq.heapify(heap)

    return vocab, merges

In [27]:
# INPUT_PATH = "input.txt"
INPUT_PATH = "pubmed_filtered_corpus.txt"
VOCAB_SIZE     = 32_000    
# VOCAB_SIZE     = 1000
SPECIAL_TOKENS = MEDICAL_SPECIAL_TOKENS
RESULTS_DIR    = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# Create corpus (run once, then comment out)

# from datasets import load_dataset
# ds = load_dataset("ccdv/pubmed-summarization", split="train[:10%]")
# create_filtered_medical_corpus(ds, INPUT_PATH)

In [28]:
vocab, merges = train_medical_bpe_tokenizer(
    input_path=INPUT_PATH,
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    use_medical_regex=True,
)

Corpus: 200.9 MB
Initial vocab: 257 | Merges to do: 31,743
Unique pairs: 1,944


BPE merges: 100%|██████████| 31743/31743 [02:50<00:00, 186.60merge/s] 


In [29]:
# Convert vocab: int → str (JSON keys must be strings)
vocab_json = {str(token_id): vocab[token_id].decode("utf-8", errors="replace") 
                for token_id in vocab}

# Create reverse mapping: bytes → token_id
bytes_to_id = {v: k for k, v in vocab.items()}

# Convert merges: tuple of bytes → list of strings
merges_json = []
for pair in merges:
    left_id = bytes_to_id.get(pair[0])
    right_id = bytes_to_id.get(pair[1])
    if left_id is not None and right_id is not None:
        merges_json.append([vocab[left_id].decode("utf-8", errors="replace"), vocab[right_id].decode("utf-8", errors="replace")])

# Save as JSON
with open(f"{RESULTS_DIR}/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_json, f, indent=2, ensure_ascii=False)

with open(f"{RESULTS_DIR}/merges.json", "w", encoding="utf-8") as f:
    json.dump(merges_json, f, indent=2, ensure_ascii=False)

# Save as pickle 
with open(f"{RESULTS_DIR}/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)
with open(f"{RESULTS_DIR}/merges.pkl", "wb") as f:
    pickle.dump(merges, f)

In [30]:
class MedicalBPETokenizer:
    """
    BPE tokenizer for medical text.
    Loads vocab and merges from JSON files produced by train_medical_bpe_tokenizer.

    Usage:
        tokenizer = MedicalBPETokenizer("./results")
        ids  = tokenizer.encode("Patient treated with IL-6 inhibitors.")
        text = tokenizer.decode(ids)
    """

    def __init__(self, results_dir: str, use_medical_regex: bool = True):
        self.pattern = MEDICAL_PRE_TOKEN_REGEX if use_medical_regex else GPT2_PRE_TOKEN_REGEX

        # Load from pickle files
        with open(f"{results_dir}/vocab.pkl", "rb") as f:
            self.vocab: dict[int, bytes] = pickle.load(f)
        
        with open(f"{results_dir}/merges.pkl", "rb") as f:
            merges: list[tuple[bytes, bytes]] = pickle.load(f)

        self.bytes_to_id: dict[bytes, int] = {v: k for k, v in self.vocab.items()}

        # merge_rank[(id_a, id_b)] = priority  (lower = applied first)
        self.merge_rank: dict[tuple[int, int], int] = {}
        for rank, (a_bytes, b_bytes) in enumerate(merges):
            if a_bytes in self.bytes_to_id and b_bytes in self.bytes_to_id:
                pair = (self.bytes_to_id[a_bytes], self.bytes_to_id[b_bytes])
                self.merge_rank[pair] = rank
        print(f"Loaded : {len(self.vocab):,} tokens | {len(self.merge_rank):,} merges")


    def _apply_merges(self, token_ids: list[int]) -> list[int]:
        """Apply highest-priority merge repeatedly until none remain."""
        while len(token_ids) >= 2:
            best_rank, best_idx = float("inf"), -1
            for i in range(len(token_ids) - 1):
                rank = self.merge_rank.get((token_ids[i], token_ids[i + 1]), float("inf"))
                if rank < best_rank:
                    best_rank, best_idx = rank, i
            if best_idx == -1:
                break
            a, b = token_ids[best_idx], token_ids[best_idx + 1]
            merged_id = self.bytes_to_id.get(self.vocab[a] + self.vocab[b])
            if merged_id is None:
                break
            token_ids = token_ids[:best_idx] + [merged_id] + token_ids[best_idx + 2:]
        return token_ids
    

    def encode(self, text: str, normalize: bool = True) -> list[int]:
        if normalize:
            text = normalize_medical_text(text)
        token_ids = []
        for match in regex.finditer(self.pattern, text):
            word_bytes = match.group().encode("utf-8")
            word_ids = [self.bytes_to_id[bytes([b])] for b in word_bytes]
            token_ids.extend(self._apply_merges(word_ids))
        return token_ids
    

    def decode(self, token_ids: list[int]) -> str:
        chunks = [self.vocab[tid] for tid in token_ids if tid in self.vocab]
        return b"".join(chunks).decode("utf-8", errors="replace")
    

    def vocab_size(self) -> int:
        return len(self.vocab)

In [31]:
tokenizer = MedicalBPETokenizer(RESULTS_DIR)

Loaded : 32,000 tokens | 31,743 merges


In [38]:
# Cell: Token-level inspection
sample = " hepcidin inhibits iron transport across cell membranes , thus decreasing the accessibility of stored iron and gastrointestinal absorption of dietary iron , leading to an increased frequency of iron - restricted erythropoiesis.1416 many randomized trials examined the role of intravenous iron in addition to esas in the treatment of anemia in patients with cancer."
ids    = tokenizer.encode(sample)
tokens = [tokenizer.decode([i]) for i in ids]
print(f"Input  : {sample}")
print(f"Tokens : {tokens}")
print(f"Count  : {len(ids)} tokens")

Input  :  hepcidin inhibits iron transport across cell membranes , thus decreasing the accessibility of stored iron and gastrointestinal absorption of dietary iron , leading to an increased frequency of iron - restricted erythropoiesis.1416 many randomized trials examined the role of intravenous iron in addition to esas in the treatment of anemia in patients with cancer.
Tokens : ['hep', 'c', 'idin', ' inhibits', ' iron', ' transport', ' across', ' cell', ' membranes', ' ,', ' thus', ' decreasing', ' the', ' accessibility', ' of', ' stored', ' iron', ' and', ' gastrointestinal', ' absorption', ' of', ' dietary', ' iron', ' ,', ' leading', ' to', ' an', ' increased', ' frequency', ' of', ' iron', ' -', ' restricted', ' erythropoiesis', '.', '1416', ' many', ' randomized', ' trials', ' examined', ' the', ' role', ' of', ' intravenous', ' iron', ' in', ' addition', ' to', ' esas', ' in', ' the', ' treatment', ' of', ' anemia', ' in', ' patients', ' with', ' cancer', '.']
Count  : 59 token

In [37]:
# Cell: Token-level inspection
sample = "anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight."
ids    = tokenizer.encode(sample)
tokens = [tokenizer.decode([i]) for i in ids]
print(f"Input  : {sample}")
print(f"Tokens : {tokens}")
print(f"Count  : {len(ids)} tokens")

Input  : anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight.
Tokens : ['anth', 'rop', 'ometric', ' study', ' of', ' elementary', ' school', ' students', ' in', ' shiraz', ' revealed', ' that', ' 16', '%', ' of', ' them', ' suffer', ' from', ' malnutrition', ' and', ' low', ' body', ' weight', '.']
Count  : 24 tokens


In [39]:
sent1 = "The patient was treated with IL-6 inhibitors for rheumatoid arthritis."
ids = tokenizer.encode(sent1)
print(ids)

[84, 261, 733, 341, 1391, 325, 32, 73, 76, 45, 54, 3480, 331, 9763, 6578, 46]


In [40]:
sent = tokenizer.decode(ids)
print(sent)

The patient was treated with IL-6 inhibitors for rheumatoid arthritis.
